In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score
plt.style.use('ggplot')
sns.set_palette('husl')
%matplotlib inline

# Càrrega de dades
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print()
print(train.head())

Train: (891, 12)
Test:  (418, 11)

   PassengerId  Survived  Pclass   
0            1         0       3  \
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp   
0                            Braund, Mr. Owen Harris    male  22.0      1  \
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            

In [2]:
def netejar_dades(df):
    df = df.copy()
    
    # PRIMER — capturar els nuls originals abans d'imputar
    df['EsImmigrantPobre'] = (
        (df['Age'].isna()) & 
        (df['Cabin'].isna()) & 
        (df['Pclass'] == 3)
    ).astype(int)
    
    # 1. Extreure títol
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(
        ['Lady','Countess','Capt','Col','Don',
         'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
    )
    df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    
    # 2. Features de família PRIMER (necessitem FamilySize per imputar edat)
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['TicketSize'] = df['Ticket'].map(df['Ticket'].value_counts())
    df['GroupSize']  = df[['FamilySize','TicketSize']].max(axis=1)
    df['GroupSize_Cat'] = pd.cut(
        df['GroupSize'],
        bins=[0, 1, 4, 100],
        labels=['sol', 'optim', 'gran']
    )
    
    # 3. Imputar edat per títol + classe
    mediana_edat = df.groupby(['Title','Pclass'])['Age'].median()
    for title in df['Title'].unique():
        for pclass in [1, 2, 3]:
            mask = (df['Title'] == title) & (df['Pclass'] == pclass) & (df['Age'].isna())
            if mask.sum() > 0:
                df.loc[mask, 'Age'] = mediana_edat.get((title, pclass), df['Age'].median())
    
    # Millora: Miss de 3a classe per mida família
    mask_miss = (df['Title'] == 'Miss') & (df['Pclass'] == 3) & (df['Age'].isna())
    df.loc[mask_miss & (df['FamilySize'] >= 3), 'Age'] = 8
    df.loc[mask_miss & (df['FamilySize'] < 3), 'Age']  = 18
    
    # 4. Imputar Embarked
    df['Embarked'] = df['Embarked'].fillna('S')
    
    # 5. Coberta
    df['Deck'] = df.apply(
        lambda x: x['Cabin'][0] if pd.notna(x['Cabin']) 
        else f"U{x['Pclass']}", 
        axis=1
    )
    df['Deck_Risk'] = df['Deck'].map({
        'A': 0, 'B': 0, 'C': 1,
        'D': 0, 'E': 1, 'F': 0,
        'G': 0, 'U1': 0, 'U2': 0, 'U3': 0
    }).fillna(0)
    
    # 6. Interacció Sexe x Classe
    df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)
    
    # 7. Fare logarítmic
    df['Fare_log'] = np.log1p(df['Fare'])
    
    # 8. Banda d'edat
    df['Age_band'] = pd.cut(
        df['Age'],
        bins=[0, 12, 18, 35, 60, 100],
        labels=['nen','adolescent','jove','adult','gran']
    )
    
    # 9. Prefix ticket
    df['TicketPrefix'] = df['Ticket'].str.extract(r'^([A-Za-z./]+)', expand=False).fillna('NUM')
    df['TicketPrefix'] = df['TicketPrefix'].apply(
        lambda x: x if x in ['PC','NUM'] else 'OTHER'
    )
    
    # 10. Interaccions edat × sexe × classe
    df['Is_Child']  = (df['Age'] < 12).astype(int)
    df['Child_3rd'] = ((df['Age'] < 12) & (df['Pclass'] == 3)).astype(int)
    df['Woman_3rd'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)
    df['Man_1st']   = ((df['Sex'] == 'male') & (df['Pclass'] == 1)).astype(int)
    
    # 11. Costat del vaixell
    df['Cabin_Num'] = df['Cabin'].str.extract(r'(\d+)', expand=False).astype(float)
    df['Cabin_Side'] = df['Cabin_Num'].apply(
        lambda x: 'E' if pd.notna(x) and x % 2 == 0 
        else 'B' if pd.notna(x) 
        else 'U'
    )
    df['Side_Pclass'] = df['Cabin_Side'] + '_' + df['Pclass'].astype(str)
    
    # 12. Model causal 3 fases
    df['Porta_Tancada']  = ((df['Pclass'] == 3) & (df['FamilySize'] >= 5)).astype(int)
    df['Man_WithFamily'] = ((df['Sex'] == 'male') & (df['FamilySize'] > 1)).astype(int)
    df['Man_Solo_Low']   = ((df['Sex'] == 'male') & (df['FamilySize'] == 1) & (df['Pclass'] >= 2)).astype(int)
    
    # 13. Coberta C risc homes 1a
    df['Man1st_CobertaC'] = ((df['Sex'] == 'male') & (df['Pclass'] == 1) & (df['Deck'] == 'C')).astype(int)
    df['Man1st_SafeDeck'] = ((df['Sex'] == 'male') & (df['Pclass'] == 1) & (df['Deck'].isin(['D','E']))).astype(int)
    
    return df

# Aplicar
train_net = netejar_dades(train)
test_net  = netejar_dades(test)
print("Neteja completada!")
print(f"Features disponibles: {len(train_net.columns)}")

Neteja completada!
Features disponibles: 36


In [3]:
# Holdout set - separar ABANS de tot
train_real, holdout = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['Survived']
)

train_real_net = netejar_dades(train_real)
holdout_net    = netejar_dades(holdout)

print(f"Train real: {len(train_real)} passatgers")
print(f"Holdout:    {len(holdout)} passatgers")
print(f"Supervivència train: {train_real['Survived'].mean():.1%}")
print(f"Supervivència holdout: {holdout['Survived'].mean():.1%}")

# Features base V2 (millor Kaggle: 0.7823)
features_v2 = [
    'Sex_Pclass', 'Title', 'Fare_log',
    'Pclass', 'FamilySize', 'TicketSize'
]

# Funció d'avaluació amb holdout
def avaluar(features, nom, model=None):
    if model is None:
        model = RandomForestClassifier(
            n_estimators=300, max_depth=4,
            min_samples_leaf=5, random_state=42
        )
    
    df_tr = train_real_net.copy()
    df_ho = holdout_net.copy()
    
    cols_cat = [c for c in ['Sex_Pclass','Title','Age_band','TicketPrefix',
                             'Embarked','Deck','GroupSize_Cat','Cabin_Side','Side_Pclass'] 
                if c in features]
    
    for col in cols_cat:
        le = LabelEncoder()
        combined = pd.concat([df_tr[col], df_ho[col]]).astype(str)
        le.fit(combined)
        df_tr[col] = le.transform(df_tr[col].astype(str))
        df_ho[col] = le.transform(df_ho[col].astype(str))
    
    X_tr = df_tr[features]
    y_tr = df_tr['Survived']
    X_ho = df_ho[features]
    y_ho = df_ho['Survived']
    
    cv = cross_val_score(model, X_tr, y_tr, cv=5, scoring='accuracy').mean()
    model.fit(X_tr, y_tr)
    ho = accuracy_score(y_ho, model.predict(X_ho))
    
    print(f"{nom:45s} CV: {cv:.4f}  Holdout: {ho:.4f}")
    return model, cv, ho

# Avaluar base
avaluar(features_v2, 'Base V2')

Train real: 712 passatgers
Holdout:    179 passatgers
Supervivència train: 38.3%
Supervivència holdout: 38.5%
Base V2                                       CV: 0.8189  Holdout: 0.8101


(RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                        random_state=42),
 0.8188909681867429,
 0.8100558659217877)

In [4]:
# Funció per generar submission
def generar_submission(features, nom_fitxer, model=None):
    if model is None:
        model = RandomForestClassifier(
            n_estimators=300, max_depth=4,
            min_samples_leaf=5, random_state=42
        )
    
    # Entrenar amb TOT el train (no només train_real)
    df_tr = train_net.copy()
    df_te = test_net.copy()
    
    cols_cat = [c for c in ['Sex_Pclass','Title','Age_band','TicketPrefix',
                             'Embarked','Deck','GroupSize_Cat','Cabin_Side','Side_Pclass'] 
                if c in features]
    
    for col in cols_cat:
        le = LabelEncoder()
        combined = pd.concat([df_tr[col], df_te[col]]).astype(str)
        le.fit(combined)
        df_tr[col] = le.transform(df_tr[col].astype(str))
        df_te[col] = le.transform(df_te[col].astype(str))
    
    # Imputar NaN
    df_te['Fare_log'] = df_te['Fare_log'].fillna(df_te['Fare_log'].median())
    
    model.fit(df_tr[features], df_tr['Survived'])
    predictions = model.predict(df_te[features])
    
    submission = pd.DataFrame({
        'PassengerId': test['PassengerId'],
        'Survived':    predictions
    })
    
    submission.to_csv(f'../submissions/{nom_fitxer}', index=False)
    print(f"Submission guardada: {nom_fitxer}")
    print(submission['Survived'].value_counts())
    return submission

print("Pipeline complet i llest!")

Pipeline complet i llest!


In [5]:
# Avaluar totes les combinacions
avaluar(features_v2, 'Base V2')
avaluar(features_v2 + ['Porta_Tancada'],   '+ Porta_Tancada')
avaluar(features_v2 + ['Man_WithFamily'],  '+ Man_WithFamily')
avaluar(features_v2 + ['Man1st_CobertaC'], '+ Man1st_CobertaC')
avaluar(features_v2 + ['Man_Solo_Low'],    '+ Man_Solo_Low')
avaluar(features_v2 + ['Porta_Tancada', 'Man_WithFamily', 'Man1st_CobertaC'], '+ Model causal complet')

Base V2                                       CV: 0.8189  Holdout: 0.8101
+ Porta_Tancada                               CV: 0.8217  Holdout: 0.8101
+ Man_WithFamily                              CV: 0.8217  Holdout: 0.8212
+ Man1st_CobertaC                             CV: 0.8231  Holdout: 0.8101
+ Man_Solo_Low                                CV: 0.8189  Holdout: 0.8101
+ Model causal complet                        CV: 0.8175  Holdout: 0.8212


(RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                        random_state=42),
 0.8174923667881414,
 0.8212290502793296)

In [6]:
generar_submission(
    features_v2 + ['Man_WithFamily'],
    'submission_man_withfamily.csv'
)



Submission guardada: submission_man_withfamily.csv
Survived
0    270
1    148
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
...,...,...
413,1305,0
414,1306,1
415,1307,0
416,1308,0


In [7]:
# Homes que van sobreviure contra les probabilitats
homes = train[train['Sex']=='male'].copy()
homes['Deck'] = homes['Cabin'].str[0].fillna('U')
homes['FamilySize'] = homes['SibSp'] + homes['Parch'] + 1

# Grups de risc alt que van sobreviure
print("Homes 1a coberta C supervivents vs morts:")
print(homes[(homes['Pclass']==1) & (homes['Deck']=='C')].groupby('Survived')['Age'].describe())
print()
print("Homes 2a preu baix supervivents vs morts:")
print(homes[(homes['Pclass']==2) & (homes['Fare']<=13)].groupby('Survived')['Age'].describe())

Homes 1a coberta C supervivents vs morts:
          count       mean        std    min    25%   50%    75%   max
Survived                                                              
0          18.0  40.694444  13.129602  18.00  31.50  41.0  48.50  64.0
1           8.0  32.365000  17.820135   0.92  23.75  33.0  48.25  52.0

Homes 2a preu baix supervivents vs morts:
          count       mean        std   min    25%   50%   75%   max
Survived                                                            
0          46.0  32.021739  12.433805  16.0  23.25  29.5  36.0  70.0
1           5.0  37.600000  15.946787  19.0  31.00  34.0  42.0  62.0


In [8]:
# Homes que van sobreviure "contra les probabilitats"
# 1a coberta C i 2a preu baix

print("Homes 1a coberta C - família vs sols:")
homes_1a_C = train[
    (train['Sex']=='male') & 
    (train['Pclass']==1) & 
    (train['Cabin'].str.startswith('C', na=False))
].copy()
homes_1a_C['FamilySize'] = homes_1a_C['SibSp'] + homes_1a_C['Parch'] + 1
print(homes_1a_C.groupby(['FamilySize','Survived']).size().unstack(fill_value=0))
print()

print("Homes 2a preu baix - família vs sols:")
homes_2a_barats = train[
    (train['Sex']=='male') & 
    (train['Pclass']==2) & 
    (train['Fare']<=13)
].copy()
homes_2a_barats['FamilySize'] = homes_2a_barats['SibSp'] + homes_2a_barats['Parch'] + 1
print(homes_2a_barats.groupby(['FamilySize','Survived']).size().unstack(fill_value=0))

Homes 1a coberta C - família vs sols:
Survived    0  1
FamilySize      
1           8  6
2           8  3
3           3  1
4           0  1
6           2  0

Homes 2a preu baix - família vs sols:
Survived     0  1
FamilySize       
1           50  6
2            1  0
4            1  0


In [9]:
# Homes 3a que van sobreviure - eren sols o amb família?
homes_3a_surv = train[
    (train['Sex']=='male') & 
    (train['Pclass']==3) & 
    (train['Survived']==1)
].copy()
homes_3a_surv['FamilySize'] = homes_3a_surv['SibSp'] + homes_3a_surv['Parch'] + 1

homes_3a_morts = train[
    (train['Sex']=='male') & 
    (train['Pclass']==3) & 
    (train['Survived']==0)
].copy()
homes_3a_morts['FamilySize'] = homes_3a_morts['SibSp'] + homes_3a_morts['Parch'] + 1

print("Homes 3a SUPERVIVENTS - distribució família:")
print(homes_3a_surv['FamilySize'].value_counts().sort_index())
print(f"Sols: {(homes_3a_surv['FamilySize']==1).sum()} / {len(homes_3a_surv)} = {(homes_3a_surv['FamilySize']==1).mean():.1%}")
print()
print("Homes 3a MORTS - distribució família:")
print(homes_3a_morts['FamilySize'].value_counts().sort_index())
print(f"Sols: {(homes_3a_morts['FamilySize']==1).sum()} / {len(homes_3a_morts)} = {(homes_3a_morts['FamilySize']==1).mean():.1%}")

Homes 3a SUPERVIVENTS - distribució família:
FamilySize
1    32
2     5
3     8
4     1
7     1
Name: count, dtype: int64
Sols: 32 / 47 = 68.1%

Homes 3a MORTS - distribució família:
FamilySize
1     232
2      23
3      17
4       2
5       3
6      12
7       3
8       4
11      4
Name: count, dtype: int64
Sols: 232 / 300 = 77.3%


In [12]:
# Afegir feature nova a train_real_net i holdout_net
for df in [train_real_net, holdout_net, train_net, test_net]:
    df['Man1st_Sol'] = (
        (df['Sex'] == 'male') & 
        (df['Pclass'] == 1) & 
        (df['FamilySize'] == 1)
    ).astype(int)

# Ara avaluar
avaluar(features_v2 + ['Man1st_Sol'], '+ Man1st_Sol')
avaluar(features_v2 + ['Man1st_Sol', 'Man1st_CobertaC'], '+ Man1st_Sol + CobertaC')

+ Man1st_Sol                                  CV: 0.8217  Holdout: 0.8101
+ Man1st_Sol + CobertaC                       CV: 0.8175  Holdout: 0.8101


(RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                        random_state=42),
 0.8175317640106373,
 0.8100558659217877)

In [13]:
# Supervivència de 3a classe per port d'embarcament
print("3a classe - supervivència per port:")
print(train[train['Pclass']==3].groupby('Embarked')['Survived'].agg(['sum','count','mean']))

3a classe - supervivència per port:
          sum  count      mean
Embarked                      
C          25     66  0.378788
Q          27     72  0.375000
S          67    353  0.189802


In [14]:
print("2a classe - supervivència per port:")
print(train[train['Pclass']==2].groupby('Embarked')['Survived'].agg(['sum','count','mean']))
print()
print("3a classe - supervivència per port:")
print(train[train['Pclass']==3].groupby('Embarked')['Survived'].agg(['sum','count','mean']))

2a classe - supervivència per port:
          sum  count      mean
Embarked                      
C           9     17  0.529412
Q           2      3  0.666667
S          76    164  0.463415

3a classe - supervivència per port:
          sum  count      mean
Embarked                      
C          25     66  0.378788
Q          27     72  0.375000
S          67    353  0.189802


In [15]:
# Afegir feature nova
for df in [train_real_net, holdout_net, train_net, test_net]:
    df['Southampton_3rd'] = (
        (df['Embarked'] == 'S') & 
        (df['Pclass'] == 3)
    ).astype(int)
    df['Southampton_2nd'] = (
        (df['Embarked'] == 'S') & 
        (df['Pclass'] == 2)
    ).astype(int)

# Avaluar
avaluar(features_v2 + ['Southampton_3rd'], '+ Southampton_3rd')
avaluar(features_v2 + ['Southampton_2nd'], '+ Southampton_2nd')
avaluar(features_v2 + ['Southampton_3rd', 'Southampton_2nd'], '+ Southampton 2a+3a')

+ Southampton_3rd                             CV: 0.8175  Holdout: 0.8045
+ Southampton_2nd                             CV: 0.8203  Holdout: 0.8101
+ Southampton 2a+3a                           CV: 0.8217  Holdout: 0.8045


(RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                        random_state=42),
 0.8216881709839458,
 0.8044692737430168)

In [16]:
# Qui eren els de Southampton 3a que van sobreviure?
surv_s_3a = train[
    (train['Pclass']==3) & 
    (train['Embarked']=='S') & 
    (train['Survived']==1)
].copy()
surv_s_3a['FamilySize'] = surv_s_3a['SibSp'] + surv_s_3a['Parch'] + 1
print(surv_s_3a[['Name','Sex','Age','FamilySize']].to_string())


                                                          Name     Sex   Age  FamilySize
2                                       Heikkinen, Miss. Laina  female  26.0           1
8            Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  female  27.0           3
10                             Sandstrom, Miss. Marguerite Rut  female   4.0           3
25   Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson)  female  38.0           7
68                             Andersson, Miss. Erna Alexandra  female  17.0           7
74                                               Bing, Mr. Lee    male  32.0           1
79                                    Dowdell, Miss. Elizabeth  female  30.0           1
81                                 Sheerlinck, Mr. Jan Baptist    male  29.0           1
85     Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)  female  33.0           4
106                           Salkjelsvik, Miss. Anna Kristine  female  21.0           1
107                  

In [17]:
# Les portes tancades correlacionen amb el port?
# Si Southampton = portes tancades, les famílies grans de Southampton haurien de morir totes
print("Famílies grans (5+) de 3a per port:")
train_copy = train.copy()
train_copy['FamilySize'] = train_copy['SibSp'] + train_copy['Parch'] + 1

grans_3a = train_copy[
    (train_copy['Pclass']==3) & 
    (train_copy['FamilySize']>=5)
]
print(grans_3a.groupby('Embarked')['Survived'].agg(['sum','count','mean']))
print()

# I famílies petites
petites_3a = train_copy[
    (train_copy['Pclass']==3) & 
    (train_copy['FamilySize']<5)
]
print("Famílies petites (<5) de 3a per port:")
print(petites_3a.groupby('Embarked')['Survived'].agg(['sum','count','mean']))

Famílies grans (5+) de 3a per port:
          sum  count      mean
Embarked                      
Q           0      5  0.000000
S           4     49  0.081633

Famílies petites (<5) de 3a per port:
          sum  count      mean
Embarked                      
C          25     66  0.378788
Q          27     67  0.402985
S          63    304  0.207237


In [18]:
# Errors del model sobre tot el train
df_tr_full = train_net.copy()

for col in ['Sex_Pclass', 'Title']:
    le = LabelEncoder()
    combined = pd.concat([df_tr_full[col], test_net[col]]).astype(str)
    le.fit(combined)
    df_tr_full[col] = le.transform(df_tr_full[col].astype(str))

model_err = RandomForestClassifier(
    n_estimators=300, max_depth=4,
    min_samples_leaf=5, random_state=42
)
model_err.fit(df_tr_full[features_v2], df_tr_full['Survived'])
preds = model_err.predict(df_tr_full[features_v2])

errors = train_net[preds != train_net['Survived'].values].copy()
errors['FamilySize'] = errors['SibSp'] + errors['Parch'] + 1
errors['Deck'] = errors['Cabin'].str[0].fillna('U')

print(f"Total errors: {len(errors)}")
print()
print("Errors per grup:")
print(errors.groupby(['Pclass','Sex'])['Survived'].agg(['sum','count']))

Total errors: 146

Errors per grup:
               sum  count
Pclass Sex               
1      female    0      3
       male     42     42
2      female    0      6
       male      8      8
3      female   13     50
       male     37     37


In [20]:
# Cabines dels homes de 1a classe morts - sense nulls
homes_1a_morts = train[
    (train['Pclass']==1) & 
    (train['Sex']=='male') & 
    (train['Survived']==0) &
    (train['Cabin'].notna())
].copy()

print(f"Total homes 1a morts amb cabina: {len(homes_1a_morts)}")
print()
print(homes_1a_morts[['Name','Age','Cabin']].to_string())

Total homes 1a morts amb cabina: 56

                                           Name   Age        Cabin
6                       McCarthy, Mr. Timothy J  54.0          E46
27               Fortune, Mr. Charles Alexander  19.0  C23 C25 C27
54               Ostby, Mr. Engelhart Cornelius  65.0          B30
62                  Harris, Mr. Henry Birkhardt  45.0          C83
92                  Chaffee, Mr. Herbert Fuller  46.0          E31
96                    Goldschmidt, Mr. George B  71.0           A5
102                   White, Mr. Richard Frasar  21.0          D26
110              Porter, Mr. Walter Chamberlain  47.0         C110
118                    Baxter, Mr. Quigg Edmond  24.0      B58 B60
124                 White, Mr. Percival Wayland  54.0          D26
137                 Futrelle, Mr. Jacques Heath  37.0         C123
139                          Giglio, Mr. Victor  24.0          B86
170                   Van der hoef, Mr. Wyckoff  61.0          B19
174                     S

In [21]:
# % de morts de 1a classe per lletra de coberta
homes_1a_morts = train[
    (train['Pclass']==1) & 
    (train['Sex']=='male') &
    (train['Cabin'].notna()) &
    (train['Survived']==0)
].copy()
homes_1a_morts['Deck'] = homes_1a_morts['Cabin'].str[0]

total_morts = len(homes_1a_morts)
print(f"Total homes 1a morts amb cabina: {total_morts}")
print()
print("Distribució per coberta:")
dist = homes_1a_morts['Deck'].value_counts()
pct  = (dist / total_morts * 100).round(1)
print(pd.DataFrame({'Morts': dist, '%': pct}))

Total homes 1a morts amb cabina: 56

Distribució per coberta:
      Morts     %
Deck             
C        21  37.5
B        12  21.4
A         8  14.3
E         7  12.5
D         7  12.5
T         1   1.8


In [22]:
# Números de cabina C dels homes 1a que van morir
homes_1a_C_morts = train[
    (train['Pclass']==1) & 
    (train['Sex']=='male') & 
    (train['Survived']==0) &
    (train['Cabin'].str.startswith('C', na=False))
].copy()

print(f"Homes 1a coberta C morts: {len(homes_1a_C_morts)}")
print()
print(homes_1a_C_morts[['Name','Cabin']].to_string())

Homes 1a coberta C morts: 21

                                           Name        Cabin
27               Fortune, Mr. Charles Alexander  C23 C25 C27
62                  Harris, Mr. Henry Birkhardt          C83
110              Porter, Mr. Walter Chamberlain         C110
137                 Futrelle, Mr. Jacques Heath         C123
245                 Minahan, Dr. William Edward          C78
252                   Stead, Mr. William Thomas          C87
273                       Natsch, Mr. Charles H         C118
331                         Partner, Mr. Austen         C124
332                   Graham, Mr. George Edward          C91
336                   Pears, Mr. Thomas Clinton           C2
351      Williams-Lambert, Mr. Fletcher Fellows         C128
377                   Widener, Mr. Harry Elkins          C82
438                           Fortune, Mr. Mark  C23 C25 C27
452             Foreman, Mr. Benjamin Laventall         C111
492                  Molson, Mr. Harry Markland        

In [23]:
for df in [train_real_net, holdout_net, train_net, test_net]:
    df['Boiler_Risk'] = (
        (df['Deck'].isin(['C','D'])) & 
        (df['Pclass']==1)
    ).astype(int)

avaluar(features_v2 + ['Boiler_Risk'], '+ Boiler_Risk')
avaluar(features_v2 + ['Boiler_Risk', 'Man1st_CobertaC'], '+ Boiler_Risk + CobertaC')

+ Boiler_Risk                                 CV: 0.8161  Holdout: 0.8101
+ Boiler_Risk + CobertaC                      CV: 0.8147  Holdout: 0.8101


(RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                        random_state=42),
 0.8146754653796908,
 0.8100558659217877)